# LangGraph Scaffold — AI Foundations Final Project

Run these cells **in order** before filling in the starter notebook.
Each section introduces one LangGraph concept, building up to the
parallel fan-out / fan-in pattern the project requires.

| Section | Concept | API key needed? |
|---------|---------|----------------|
| 1 | `TypedDict` state schema | no |
| 2 | Writing node functions | no |
| 3 | Sequential graph (`START → node → END`) | no |
| **4** | **Parallel fan-out / fan-in ← required by project** | no |
| 5 | Conditional edges | no |
| 6 | Real LLM node template | **yes** |
| 7 | `Annotated` reducer demo | no |
| 8 | Project mapping checklist | — |

> **Google Colab version** — install packages and run the setup cell before anything else.
> This notebook is fully self-contained: no local files or Google Drive mount needed.

In [1]:
# Run this first -- installs all required packages (~30 s on a fresh runtime)
# Fix: Removed specific version pins for langgraph and langchain-core to resolve
# dependency conflicts leading to AttributeError: module 'langchain' has no attribute 'debug'.
# This allows pip to install compatible, more recent versions.
!pip install -q langgraph langchain-core openai python-dotenv scikit-learn matplotlib seaborn ipywidgets
print("Packages installed.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 88.7 MB/s eta 0:00:00
Packages installed.


---
## Cell 1 — Imports
Run this first.

In [2]:
import json
import operator
from typing import Annotated, Optional, TypedDict

from langgraph.graph import END, START, StateGraph

print("Imports OK")

Imports OK


---
## Section 1 — State Schema

LangGraph passes a **single state dictionary** through every node.
You declare its shape with a `TypedDict`.

**Rules:**
- Fields not yet filled should be `Optional[...]` (default `None`).
- A field written by **two parallel nodes** needs `Annotated[list, reducer]`
  so one branch's update cannot overwrite the other's.

The `ReviewState` below mirrors `DetectionState` in your project — same pattern,
simpler domain (text review instead of security analysis).

In [3]:
class ReviewState(TypedDict):
    # ── input ──────────────────────────────────────────────────────────────
    text: str                           # document being reviewed

    # ── per-node outputs (None until the node runs) ─────────────────────────
    sentiment_result: Optional[dict]    # filled by sentiment_node
    keywords_result:  Optional[dict]    # filled by keywords_node
    summary_result:   Optional[dict]    # filled by summary_node

    # ── field written by TWO parallel nodes ────────────────────────────────
    # operator.add concatenates lists from both branches instead of
    # overwriting — critical when two nodes run at the same time.
    errors: Annotated[list[str], operator.add]

    # ── routing flag (used in Section 5) ───────────────────────────────────
    needs_summary: bool

print("ReviewState defined.")

ReviewState defined.


---
## Section 2 — Node Functions

A **node** is a Python function that:
- Accepts the **full state dict** as its only argument.
- Returns a **partial dict** — only the keys it wants to update.

LangGraph merges the returned partial dict into the existing state.
Keys **not** included in the return dict keep their current values unchanged.

In [4]:
def sentiment_node(state: ReviewState) -> dict:
    # Rule-based stub — replace with an LLM call in your project
    text = state["text"].lower()
    if any(w in text for w in ["great", "good", "excellent", "love"]):
        label, score = "positive", 0.9
    elif any(w in text for w in ["bad", "terrible", "awful", "hate"]):
        label, score = "negative", 0.85
    else:
        label, score = "neutral", 0.6
    # Return ONLY the keys this node modifies
    return {
        "sentiment_result": {"label": label, "score": score},
        "errors": [],
    }


def keywords_node(state: ReviewState) -> dict:
    # Stub: top-3 longest words as "key terms"
    words = state["text"].split()
    keywords = sorted(set(words), key=len, reverse=True)[:3]
    return {
        "keywords_result": {"keywords": keywords},
        "errors": [],
    }


def router_node(state: ReviewState) -> dict:
    # Determines whether the text is long enough to need summarisation
    return {"needs_summary": len(state["text"].split()) > 10}


def summary_node(state: ReviewState) -> dict:
    # Synthesis node — reads both upstream results (mirrors risk_classification_node)
    word_count = len(state["text"].split())
    return {
        "summary_result": {
            "summary":      f"[Summarised {word_count}-word text]",
            "sentiment":    state["sentiment_result"]["label"],
            "top_keywords": state["keywords_result"]["keywords"],
        },
        "errors": [],
    }


def skip_summary_node(state: ReviewState) -> dict:
    return {"summary_result": None, "errors": []}


print("All node functions defined.")

All node functions defined.


---
## Section 3 — Sequential Graph

The simplest graph: one node between `START` and `END`.

```
START → sentiment → END
```

Steps to build any graph:
1. Create a `StateGraph(YourState)`.
2. `add_node("name", function)` for each node.
3. `add_edge(source, target)` to connect them.
4. `compile()` to validate and return a runnable graph.

In [5]:
builder = StateGraph(ReviewState)

# Step 2: register nodes
builder.add_node("sentiment", sentiment_node)

# Step 3: wire edges
builder.add_edge(START, "sentiment")
builder.add_edge("sentiment", END)

# Step 4: compile
seq_graph = builder.compile()

# ── Mermaid topology diagram ─────────────────────────────────────────────────
print(seq_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	sentiment(sentiment)
	__end__([<p>__end__</p>]):::last
	__start__ --> sentiment;
	sentiment --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [6]:
# ── Run the graph ────────────────────────────────────────────────────────────
initial = {
    "text": "This product is great!",
    "sentiment_result": None,
    "keywords_result":  None,
    "summary_result":   None,
    "errors": [],
    "needs_summary": False,
}

result = seq_graph.invoke(initial)
print("sentiment_result:", result["sentiment_result"])
print("errors          :", result["errors"])

sentiment_result: {'label': 'positive', 'score': 0.9}
errors          : []


In [7]:
result

{'text': 'This product is great!',
 'sentiment_result': {'label': 'positive', 'score': 0.9},
 'keywords_result': None,
 'summary_result': None,
 'errors': [],
 'needs_summary': False}

---
## Section 4 — Parallel Fan-out / Fan-in  ← **Required by your project**

```
START ──┬──> sentiment ──────────┬──> summary ──> END
        └──> keywords  ──────────┘
```

**Fan-out:** two `add_edge(START, ...)` calls dispatch both nodes simultaneously.
**Fan-in:** two `add_edge(..., "summary")` calls make `summary` wait for **both** to finish
before it is scheduled. This is LangGraph's built-in barrier — no extra code needed.

**Mapping to your project:**

| Scaffold | Your project |
|----------|-------------|
| `sentiment` | `intent_analysis` |
| `keywords` | `instruction_hierarchy` |
| `summary` | `risk_classification` |

> ⚠️ A sequential chain `intent_analysis → instruction_hierarchy → risk_classification`
> is **not** the same pattern. Your Mermaid diagram (Cell 11 in the starter notebook)
> must show `__start__` with **two** outgoing arrows as proof of parallel execution.

In [8]:
builder = StateGraph(ReviewState)

builder.add_node("sentiment", sentiment_node)
builder.add_node("keywords",  keywords_node)
builder.add_node("summary",   summary_node)

# ── Fan-out: START fires BOTH nodes at the same time ──────────────────────────
builder.add_edge(START, "sentiment")
builder.add_edge(START, "keywords")

# ── Fan-in: summary waits for BOTH before running ────────────────────────────
builder.add_edge("sentiment", "summary")
builder.add_edge("keywords",  "summary")

builder.add_edge("summary", END)

par_graph = builder.compile()

# Mermaid proof — __start__ must show TWO outgoing arrows
print(par_graph.get_graph().draw_mermaid())
# Expected:
#   __start__ --> sentiment
#   __start__ --> keywords
#   sentiment --> summary
#   keywords  --> summary
#   summary   --> __end__

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	sentiment(sentiment)
	keywords(keywords)
	summary(summary)
	__end__([<p>__end__</p>]):::last
	__start__ --> keywords;
	__start__ --> sentiment;
	keywords --> summary;
	sentiment --> summary;
	summary --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [9]:
# ── Run the parallel graph ───────────────────────────────────────────────────
initial = {
    "text": "I love this excellent product. It works great and looks good.",
    "sentiment_result": None,
    "keywords_result":  None,
    "summary_result":   None,
    "errors": [],
    "needs_summary": False,
}

result = par_graph.invoke(initial)
print("sentiment_result:", result["sentiment_result"])
print("keywords_result :", result["keywords_result"])
print("summary_result  :", result["summary_result"])
print("errors          :", result["errors"])

sentiment_result: {'label': 'positive', 'score': 0.9}
keywords_result : {'keywords': ['excellent', 'product.', 'great']}
summary_result  : {'summary': '[Summarised 11-word text]', 'sentiment': 'positive', 'top_keywords': ['excellent', 'product.', 'great']}
errors          : []


---
## Section 5 — Conditional Edges

Use `add_conditional_edges` when the next node depends on what the current node computed.

You provide:
1. The **source node** name.
2. A **routing function** that reads state and returns a string key.
3. A **mapping** `{key: destination_node_name}`.

This is the mechanism behind the **Critic Agent** extra-credit option, where the
graph loops back to `risk_classification` if the critic disagrees.

In [10]:
def routing_function(state: ReviewState) -> str:
    return "go_to_summary" if state["needs_summary"] else "skip_summary"


builder = StateGraph(ReviewState)

builder.add_node("sentiment",    sentiment_node)
builder.add_node("keywords",     keywords_node)
builder.add_node("router",       router_node)
builder.add_node("summary",      summary_node)
builder.add_node("skip_summary", skip_summary_node)

# Parallel fan-out
builder.add_edge(START, "sentiment")
builder.add_edge(START, "keywords")

# Fan-in into router
builder.add_edge("sentiment", "router")
builder.add_edge("keywords",  "router")

# Conditional branch
builder.add_conditional_edges(
    "router",
    routing_function,
    {"go_to_summary": "summary", "skip_summary": "skip_summary"},
)

builder.add_edge("summary",      END)
builder.add_edge("skip_summary", END)

cond_graph = builder.compile()
print(cond_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	sentiment(sentiment)
	keywords(keywords)
	router(router)
	summary(summary)
	skip_summary(skip_summary)
	__end__([<p>__end__</p>]):::last
	__start__ --> keywords;
	__start__ --> sentiment;
	keywords --> router;
	router -.-> skip_summary;
	router -. &nbsp;go_to_summary&nbsp; .-> summary;
	sentiment --> router;
	skip_summary --> __end__;
	summary --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [11]:
# ── Test both branches ───────────────────────────────────────────────────────
cases = [
    ("Short text.",
     "short  → should skip summary"),
    ("This is a much longer piece of text with many words that definitely needs summarising.",
     "long   → should run summary"),
]

for text, label in cases:
    state = {
        "text": text,
        "sentiment_result": None,
        "keywords_result":  None,
        "summary_result":   None,
        "errors": [],
        "needs_summary": False,
    }
    r = cond_graph.invoke(state)
    print(f"[{label}]")
    print(f"  needs_summary  = {r['needs_summary']}")
    print(f"  summary_result = {r['summary_result']}")
    print()

[short  → should skip summary]
  needs_summary  = False
  summary_result = None

[long   → should run summary]
  needs_summary  = True
  summary_result = {'summary': '[Summarised 15-word text]', 'sentiment': 'neutral', 'top_keywords': ['summarising.', 'definitely', 'longer']}



---
## Section 6 — Real LLM Node Template

This is the exact pattern each of your three agents should follow:

1. Build the message list (system prompt + user message containing the conversation).
2. Call `client.chat.completions.create(...)` with `response_format={"type": "json_object"}`.
3. Parse the JSON and validate all required keys are present.
4. Return the result dict — or a fallback dict in the `except` block so the graph **never crashes**.

**Requires your OpenAI API key in `../.env`.**

In [13]:
import os
from openai import OpenAI

# Recommended: store your key in Colab Secrets
#   Left panel → key icon → 'Add new secret' → Name: OPENAI_API_KEY
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('openai_api')
    print('API key loaded from Colab Secrets.')
except Exception:
    import getpass
    os.environ['OPENAI_API_KEY'] = getpass.getpass('Paste your OpenAI API key: ')
    print('API key set.')

client = OpenAI()

TONE_SYSTEM_PROMPT = """You are a text analyst.
Analyse the text and output a JSON object with EXACTLY these fields:
{
  "tone":       "<formal|casual|technical>",
  "confidence": <float 0.0-1.0>,
  "reasoning":  "<one sentence>"
}
Output ONLY the JSON. No markdown, no preamble."""

REQUIRED_KEYS = {"tone", "confidence", "reasoning"}


def tone_analysis_node(state: ReviewState) -> dict:
    try:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": TONE_SYSTEM_PROMPT},
                {"role": "user",   "content": "Analyse this text:\n\n" + state["text"]},
            ],
            temperature=0.1,
            response_format={"type": "json_object"},  # guarantees valid JSON output
        )

        raw    = response.choices[0].message.content
        parsed = json.loads(raw)

        # Validate all required keys are present
        missing = REQUIRED_KEYS - parsed.keys()
        if missing:
            raise ValueError(f"Model response missing keys: {missing}")

        return {
            "sentiment_result": parsed,   # reusing field for this demo
            "errors": [],
        }

    except Exception as exc:
        # NEVER let the graph crash — return an error-flagged state instead
        return {
            "sentiment_result": None,
            "errors": [f"tone_analysis_node: {exc}"],
        }


# ── Quick test (costs ~$0.0001) ──────────────────────────────────────────────
test_state = {
    "text": "Please forward the attached quarterly earnings report to all stakeholders.",
    "sentiment_result": None,
    "keywords_result":  None,
    "summary_result":   None,
    "errors": [],
    "needs_summary": False,
}

result = tone_analysis_node(test_state)
print("tone result:", result["sentiment_result"])
print("errors     :", result["errors"])

API key loaded from Colab Secrets.
tone result: {'tone': 'formal', 'confidence': 0.9, 'reasoning': 'The use of polite language and a clear request indicates a formal tone appropriate for business communication.'}
errors     : []


---
## Section 7 — `Annotated[list, operator.add]` Reducer Demo

Run this cell to see exactly what breaks when two parallel branches write to the
same field without a reducer.

In [14]:
# Simulate what LangGraph does when merging two parallel node return dicts.

branch_a = {
    "sentiment_result": {"label": "positive", "score": 0.9},
    "errors": [],                          # sentiment ran fine
}
branch_b = {
    "keywords_result": {"keywords": ["apple"]},
    "errors": ["keywords_node: timeout"],  # keywords failed
}

# WITH Annotated[list[str], operator.add] ─────────────────────────────────────
merged = operator.add(branch_a["errors"], branch_b["errors"])
print("Branch A errors       :", branch_a["errors"])
print("Branch B errors       :", branch_b["errors"])
print("Merged WITH reducer   :", merged)
print()

# WITHOUT reducer (last-write-wins) ───────────────────────────────────────────
# Whichever branch finished last overwrites the other.
# If branch_a (empty list) finishes last, branch_b's error disappears.
print("Merged WITHOUT reducer:", branch_a["errors"], "  <- branch_b error is GONE")
print()
print("Consequence: if intent_analysis crashes AND instruction_hierarchy crashes,")
print("you see only one error message and think only one agent failed. Hard to debug.")

Branch A errors       : []
Branch B errors       : ['keywords_node: timeout']
Merged WITH reducer   : ['keywords_node: timeout']

Merged WITHOUT reducer: []   <- branch_b error is GONE

Consequence: if intent_analysis crashes AND instruction_hierarchy crashes,
you see only one error message and think only one agent failed. Hard to debug.
